# ARC-AGI-3 Transformer agent (inference notebook)

This notebook adapts the provided Kaggle ARC-AGI-3 scaffold: the input is a sequence of `FrameData` observations and the output is a `GameAction` from `MyAgent.choose_action`. It preserves the competition rerun setup and the local `submission.parquet` schema.

The Target v3 diagrams describe an ARC grid-to-grid task solver, while this notebook's provided environment is interactive ARC-AGI-3. Accordingly, the transformer below is an action-policy adapter, not a trained grid-transformation solver. It loads a compatible checkpoint from `/kaggle/input` when supplied and otherwise falls back to the original random-action policy. A random-initialized policy is not presented as a trained solver.

The checkpoint model expects `state_dict`, optional `actions` (ordered action enum values), and `config` (`d_model`, `heads`, `layers`, `max_tokens`).

In [ ]:
# Offline Kaggle wheels supplied with the competition.
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
import os, random, time
from typing import Any
import numpy as np
import torch
import torch.nn as nn
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

class ActionTransformer(nn.Module):
    # Color/token sequence encoder plus action and (x,y) heads.
    def __init__(self, n_actions, d_model=128, heads=4, layers=2, max_tokens=1024, vocab=17):
        super().__init__(); self.max_tokens=max_tokens
        self.token=nn.Embedding(vocab,d_model); self.pos=nn.Embedding(max_tokens,d_model)
        enc=nn.TransformerEncoderLayer(d_model,heads,4*d_model,batch_first=True,dropout=0.0)
        self.encoder=nn.TransformerEncoder(enc,layers,enable_nested_tensor=False)
        self.action=nn.Linear(d_model,n_actions); self.xy=nn.Linear(d_model,2*64)
    def forward(self,tokens):
        p=torch.arange(tokens.shape[1],device=tokens.device)
        h=self.encoder(self.token(tokens)+self.pos(p)[None])
        pooled=h.mean(1); return self.action(pooled), self.xy(pooled).reshape(-1,2,64)

def _grid_from_frame(frame):
    # FrameData versions differ; inspect common fields and accept only a rectangular color grid.
    queue=[frame]; seen=set()
    while queue:
        obj=queue.pop(0)
        if id(obj) in seen: continue
        seen.add(id(obj))
        if isinstance(obj,dict):
            queue.extend(obj.values())
        elif isinstance(obj,(list,tuple,np.ndarray)):
            try:
                a=np.asarray(obj)
                if a.ndim==2 and a.size and a.dtype.kind in 'biuf' and np.isfinite(a).all() and a.min()>=0 and a.max()<=15:
                    return a.astype(np.int64)
            except (TypeError,ValueError): pass
        else:
            for name in ('grid','frame','observation','board','data','pixels'):
                try: value=getattr(obj,name)
                except Exception: continue
                if value is not None: queue.append(value)
    return np.zeros((1,1),dtype=np.int64)

class MyAgent(Agent):
    MAX_ACTIONS=float('inf')
    def __init__(self,*args:Any,**kwargs:Any)->None:
        super().__init__(*args,**kwargs)
        self.rng=random.Random(time.time_ns() ^ (hash(self.game_id)&0xffffffff))
        self.device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.actions=[a for a in GameAction if a is not GameAction.RESET]
        self.policy=None
        # Optional Kaggle dataset model file; do not silently use random weights as a learned policy.
        ckpt=os.getenv('ARC_TRANSFORMER_CHECKPOINT')
        if not ckpt:
            import glob
            hits=glob.glob('/kaggle/input/**/*.pt',recursive=True)+glob.glob('/kaggle/input/**/*.pth',recursive=True)
            ckpt=hits[0] if hits else None
        if ckpt and os.path.isfile(ckpt):
            try:
                blob=torch.load(ckpt,map_location=self.device,weights_only=False)
                cfg=blob.get('config',{})
                saved=blob.get('actions',[a.value for a in self.actions])
                current=[a.value for a in self.actions]
                if saved==current:
                    self.policy=ActionTransformer(len(self.actions),int(cfg.get('d_model',128)),int(cfg.get('heads',4)),int(cfg.get('layers',2)),int(cfg.get('max_tokens',1024))).to(self.device)
                    self.policy.load_state_dict(blob['state_dict']); self.policy.eval()
            except Exception as exc:
                print(f'Could not load transformer checkpoint {ckpt}: {exc}')
    def is_done(self,frames:list[FrameData],latest_frame:FrameData)->bool:
        return latest_frame.state is GameState.WIN
    def choose_action(self,frames:list[FrameData],latest_frame:FrameData)->GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED,GameState.GAME_OVER): return GameAction.RESET
        action=None; xy=(0,0)
        if self.policy is not None:
            g=_grid_from_frame(latest_frame)
            # Preserve grid order while bounding sequence length and memory use.
            stride=max(1,int(np.ceil(np.sqrt(g.size/ min(self.policy.max_tokens,1024)))))
            g=g[::stride,::stride].reshape(-1)[:self.policy.max_tokens]
            tok=torch.as_tensor(np.clip(g,0,9),dtype=torch.long,device=self.device)[None]
            with torch.no_grad(): logits,coords=self.policy(tok)
            idx=int(logits[0].argmax().item()); action=self.actions[idx]
            xy=(int(coords[0,0].argmax().item()),int(coords[0,1].argmax().item()))
        if action is None:
            action=self.rng.choice(self.actions)
            xy=(self.rng.randrange(64),self.rng.randrange(64))
        if action.is_complex(): action.set_data({'x':xy[0],'y':xy[1]})
        action.reasoning='Transformer policy checkpoint' if self.policy is not None else 'No compatible checkpoint; random baseline'
        return action


In [ ]:
import os, shutil
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the competition gateway and stage its agent runner.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    src='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
    dst='/kaggle/working/ARC-AGI-3-Agents'
    shutil.copytree(src,dst,dirs_exist_ok=True)
    shutil.copy('/kaggle/working/my_agent.py',dst+'/agents/templates/my_agent.py')
    # Avoid optional template imports that require packages not present offline.
    with open(dst+'/agents/__init__.py','w') as f:
        f.write('from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {\"random\": Random, \"myagent\": MyAgent}\n')
    with open(dst+'/.env','w') as f:
        f.write('SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n')
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent
else:
    # Local notebook artifact: retain the original schema. The competition rerun
    # branch above is what executes the agent against real games.
    import pandas as pd
    submission=pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score'])
    submission.to_parquet('/kaggle/working/submission.parquet',index=False)
    print('Wrote /kaggle/working/submission.parquet with columns:',list(submission.columns))